In [ ]:
# # Logistic Regression Model for Weather Prediction
#
# This notebook contains the complete step-by-step training, hyperparameter tuning, evaluation, and visualization for the **Logistic Regression** model used to predict whether it will rain tomorrow (`RainTomorrow`).
#
# ### Mathematical Concepts & Intuition
# Logistic Regression is a supervised classification algorithm that calculates the probability that a given input $X$ belongs to class 1 ($y = 1$, representing Rain tomorrow) vs class 0 ($y = 0$, representing No Rain).
#
# It applies the **sigmoid function** to a linear combination of input features:
# $$P(y = 1 | X) = \sigma(\theta^T X) = \frac{1}{1 + e^{-\theta^T X}}$$
#
# ### Hyperparameters Tuned
# To find the best model configuration, we perform a grid search over key hyperparameters:
# 1. **Solver**:
#    - `liblinear`: Highly efficient for small/medium datasets. Supports L1 and L2 regularization.
#    - `lbfgs`: A quasi-Newton optimization method. Excellent for large, continuous features.
#    - `newton-cg`: Uses second-order derivatives to optimize weights. Accurate but computationally intensive.
# 2. **Max Iterations (`max_iter`)**: The maximum number of solver iterations to guarantee optimization convergence.


In [ ]:
# Step 1: Imports and libraries
import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

# Set plotting aesthetics
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (7, 5)


In [ ]:
# ### Step 2: Data Loading
# We load the scaled features and preprocessed targets from our centralized preprocessed dataset `../data/processed_data.joblib`.


In [ ]:
# Step 2: Load the preprocessed weather dataset
PROCESSED_DATA_PATH = "../data/processed_data.joblib"

if not os.path.exists(PROCESSED_DATA_PATH):
    raise FileNotFoundError(f"Preprocessed data not found at {PROCESSED_DATA_PATH}. Please run preprocessing first.")

data = joblib.load(PROCESSED_DATA_PATH)
X_train = data['X_train']
X_test = data['X_test']
y_train = data['y_train']
y_test = data['y_test']

print(f"Dataset loaded successfully!")
print(f"Training set size: {X_train.shape[0]} samples, {X_train.shape[1]} features")
print(f"Testing set size:  {X_test.shape[0]} samples")


In [ ]:
# ### Step 3: Hyperparameter Grid Search Training
# We systematically loop through each combination of `solver` and `max_iter` to find the most accurate model configuration.


In [ ]:
# Step 3: Training with Parameter Grid Search
solvers = ['liblinear', 'lbfgs', 'newton-cg']
max_iters = [100, 500, 1000]

results = []
best_acc = 0
best_model = None

print("--- Starting Logistic Regression Tuning Loop ---")
for solver in solvers:
    for max_iter in max_iters:
        try:
            print(f"Training solver={solver:<12} | max_iter={max_iter:<5}")
            # Train model
            model = LogisticRegression(solver=solver, max_iter=max_iter, random_state=42, n_jobs=-1)
            model.fit(X_train, y_train)
            
            # Evaluate on test set
            y_pred = model.predict(X_test)
            acc = accuracy_score(y_test, y_pred)
            print(f"  --> Test Accuracy: {acc * 100:.2f}%")
            
            results.append({
                'solver': solver,
                'max_iter': max_iter,
                'accuracy': acc
            })
            
            # Retain best estimator
            if acc > best_acc:
                best_acc = acc
                best_model = model
        except Exception as e:
            print(f"  --> Failed config (solver={solver}, max_iter={max_iter}): {e}")
            continue
print("\nGrid Search Tuning Completed.")


In [ ]:
# ### Step 4: Model Evaluation & Visualizations
# We compute full classification metrics on the test set and display a styled **Confusion Matrix Heatmap**.


In [ ]:
# Step 4: Run evaluation metrics for the best Logistic Regression model
y_pred = best_model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print("=== BEST MODEL METRICS ===")
print(f"Optimal Solver:    {best_model.solver}")
print(f"Optimal Max Iter:  {best_model.max_iter}")
print(f"Test Accuracy:     {accuracy * 100:.2f}%")
print(f"Precision Score:   {precision * 100:.2f}%")
print(f"Recall Score:      {recall * 100:.2f}%")
print(f"F1 Performance:    {f1 * 100:.2f}%")

# Plot Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['No Rain', 'Rain'], yticklabels=['No Rain', 'Rain'])
plt.title('Confusion Matrix - Logistic Regression (Best Model)', fontsize=14, pad=15)
plt.ylabel('Actual Label', fontsize=12)
plt.xlabel('Predicted Label', fontsize=12)
plt.tight_layout()
plt.show()


In [ ]:
# ### Step 5: Serializing the Trained Model
# We save the optimal Logistic Regression model into our directory structure `../data/models/logistic_regression.joblib` for deployment.


In [ ]:
# Step 5: Save best model to disk
MODELS_DIR = "../data/models"
if not os.path.exists(MODELS_DIR):
    os.makedirs(MODELS_DIR)

model_path = os.path.join(MODELS_DIR, 'logistic_regression.joblib')
joblib.dump(best_model, model_path)
print(f"Best Logistic Regression model successfully saved to: {model_path}")
